# UNDERTONE - build the item pack

Run this **once**, on CPU, then upload `data/item_pack/` to Kaggle as the
`undertone-item-pack` dataset. The thirteen model notebooks attach it, so a
sweep never re-harvests audio.

## What an item is

One four-option question about **one span of one real recording**. Nothing is
inserted, spliced or synthesised - the prominence differences the categories
name are differences the speakers themselves produced. That is the whole
difference from the retired LongAudioBench tasks, whose labels were drawn with
`random.choice` before the audio existed.

| category | acoustic cause | share |
|---|---|---|
| P1 quiet | low energy relative to this recording, and a real drop below its median | 22% |
| P2 masked | overlapped by another speaker (gold, from segment timings) | 22% |
| P3 backgrounded | prosodically flat aside, while a competing value is loud and repeated | 22% |
| P4 corrected | self-repair; the corrected value is the answer | 22% |
| C1 delivered | carried by hesitation and strain - the discriminant | 12% |

## Where the options come from

Every distractor is a **real competing mention of the same quantity kind from
the same recording**, never a fabrication. That is what makes a wrong answer
diagnostic rather than merely wrong:

- `salience` - the loudest and/or most repeated competing value
- `recency` - the latest competing value
- `absent` - "not mentioned in the recording"; correct on the 10% null items

## Source

AMI Meeting Corpus (CC-BY-4.0), Mix-Headset channel. Real 30-60 min meetings
with genuine crosstalk, asides and self-repairs.

The IHM/SDM microphone pair is **not** used for the main arm. It is a channel
manipulation, and the paper plan excludes channel/room confounds explicitly
(they need an expensive codec control). It stays available as a validity-control
ablation.

## This notebook does not produce usable items

Everything it writes is `verified: false`. It becomes evidence after the leak
filter runs and after you have listened to the clips. The analysis refuses to
report unverified cells.


In [ ]:
# Pinned for this model. If `load()` fails, this cell is the first thing to change.
%pip install -q "transformers==4.57.1"
%pip install -q "accelerate>=1.0.0"
%pip install -q "librosa>=0.10.2"
%pip install -q "soundfile>=0.12.1"
%pip install -q "datasets>=2.19.0"
%pip install -q "faster-whisper>=1.0.0"
print("--- resolved versions (freeze these before the paper run) ---")
import importlib.metadata as md
for pkg in ["transformers", "accelerate", "torch", "librosa"]:
    try:
        print(f"{pkg:14s} {md.version(pkg)}")
    except md.PackageNotFoundError:
        print(f"{pkg:14s} not installed")

In [ ]:
import os, random, sys, json
import numpy as np, torch

SEED = 20260904
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Weights go to /kaggle/temp: scratch, and it does NOT count against the 20 GB
# /kaggle/working output cap. A 16-18 GB checkpoint in /kaggle/working would
# fail the commit at the end of the session.
os.environ.setdefault("HF_HOME", "/kaggle/temp/hf")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
# Long audio prompts allocate in large irregular blocks; without this the T4
# fragments and OOMs with a gigabyte nominally free.
os.environ.setdefault("PYTORCH_ALLOC_CONF", "expandable_segments:True")

for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"cuda:{i}  {p.name}  {p.total_memory/1e9:.1f} GB  sm{p.major}{p.minor}")
if torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] < 8:
    print("\nsm < 80: no bf16 compute and no flash-attention-2. "
          "Every adapter loads in fp16 for this reason.")

In [ ]:
REPO_URL = "https://github.com/DeepanIsCool/longaudiobench.git"
REPO_REF = "undertone"   # pin to a commit sha before the paper run

import subprocess, shutil, os, sys
if os.path.exists("/kaggle/working/longaudiobench"):
    shutil.rmtree("/kaggle/working/longaudiobench")
for attempt in range(3):
    rc = subprocess.call(["git", "clone", "--depth", "1", "--branch", REPO_REF,
                          REPO_URL, "/kaggle/working/longaudiobench"])
    if rc == 0:
        break
else:
    raise RuntimeError("could not clone the benchmark repo")

sys.path.insert(0, "/kaggle/working/longaudiobench")
import importlib; importlib.invalidate_caches()

from undertone import ItemPack, adapters, env, runner, scoring
print("adapters registered:", len(adapters.list_adapters()))

if env.export_hf_token():
    print("HF token resolved")
elif globals().get("GATED"):
    raise RuntimeError(
        "this model is gated and no token was found. Add a Kaggle secret named "
        "HF_TOKEN, or write the token to .hf_token at the repo root.")

hw = env.resolve_hardware()
print(f"hardware: {hw.detail}  dtype={hw.dtype}  signature={hw.signature}")
print(f"versions: {env.versions()}")
# Every result row is stamped with this signature. The analysis refuses to put
# two signatures in one table -- a benchmark whose rows came from different
# backends compares machines, not models.

In [ ]:
# ~150 MB of audio per meeting. Fifteen meetings is roughly one hour of CPU.
# English from AMI (multi-party, gold speaker turns -> the only source that can
# express P2). Hindi and Bengali come from YODAS2: long-form and CC-BY, but with
# no diarisation, so they contribute P1/P3/P4/C1 and no P2.
#
# Start English-only. YODAS2 is streamed and adds a long, unpredictable download
# to a run that has not yet proved the pipeline end to end on Kaggle; add
# "hi bn" once an English pack exists and the leak filter has run on it.
# Measured yield is ~3.5 usable proposals per AMI meeting, so 180 items needs
# roughly 50 meetings.
LANGS = "en"
N_MEETINGS = 25      # ~90 proposals at the measured 3.5/meeting

# Build the list here, not in a $(...) subshell: the subshell does not inherit
# this notebook's sys.path, so the import fails silently, --meetings gets an
# empty list, and the harvest runs over zero meetings.
from undertone.harvest.sources import AMI_SCENARIO_MEETINGS
MEETINGS = " ".join(AMI_SCENARIO_MEETINGS[:N_MEETINGS])
print(f"harvesting {N_MEETINGS} meetings: {MEETINGS[:80]}...")

!python /kaggle/working/longaudiobench/scripts/build_item_pack.py --out /kaggle/working/item_pack --audio-cache /kaggle/temp/source_audio --langs {LANGS} --target 180 --meetings {MEETINGS}

In [ ]:
# The audio-necessity gate. Two tiers, because the categories make different
# claims:
#
#   gold transcript -> gates P4 and C1. Their answers genuinely are not in the
#                      words: ASR normalises a repair away and never records
#                      hesitancy. If a text model solves one, it is not an item.
#   ASR transcript  -> gates P1, P2 and P3. These are claims about *cascaded*
#                      systems. A perfect transcript of a muttered utterance
#                      contains the answer by construction, so gating them on
#                      gold would reject the whole category and prove nothing.
#
# Both rates are reported for every category regardless of which one gates. The
# gold-leak rate we do not reject on is a limitation, and it goes in the paper
# rather than in a drawer.
from undertone import ItemPack
from undertone.harvest import asr, leakfilter

PACK_DIR = "/kaggle/working/item_pack"
pack = ItemPack.load(f"{PACK_DIR}/item_pack.jsonl")
GOLD = json.load(open(f"{PACK_DIR}/gold_transcripts.json"))

# Whisper over every recording, cached: a 30-minute transcription is minutes of
# GPU and a notebook restart must not redo it.
transcripts = asr.transcripts_for(pack, audio_root=PACK_DIR,
                                  cache_dir="/kaggle/working/asr_cache")
ASR = asr.as_text(transcripts)
print(f"transcribed {len(ASR)} recordings")

# Whisper stays resident on GPU 0 otherwise, and the leak filter's 7B text model
# then OOMs with 1.5 GB free - which is exactly how the first real run died
# *after* successfully writing the item pack.
asr.unload()
import gc; gc.collect(); torch.cuda.empty_cache()
print(f"freed the ASR engine; {torch.cuda.memory_allocated(0)/1e9:.1f} GB still allocated")

In [ ]:
# Sharper than "did a text model get it right": did the ASR write the answer
# down at all? If Whisper never transcribed the muttered words, no language
# model behind it could have found them -- audio-necessity as a fact rather than
# an inference from one model's score. This table goes in the paper.
recovery = asr.needle_recovery(transcripts, list(pack))
for row in recovery:
    print(f"{row['category']}  n={row['n']:3d}  "
          f"ASR recovered the answer in {row['recovery_rate']:.0%} of items  "
          f"-> {row['unrecoverable_rate']:.0%} unanswerable by any cascaded system")

with open("/kaggle/working/item_pack/needle_recovery.json", "w") as fh:
    json.dump(recovery, fh, indent=2)

In [ ]:
# Plug in whatever text model you have. A stronger solver is a stronger claim.
def make_solver(model_id="Qwen/Qwen2.5-7B-Instruct"):
    from transformers import AutoModelForCausalLM, AutoTokenizer
    tok = AutoTokenizer.from_pretrained(model_id)
    llm = AutoModelForCausalLM.from_pretrained(
        model_id, torch_dtype=torch.float16, device_map="auto",
        max_memory={0: "13GiB", 1: "13GiB", "cpu": "24GiB"}).eval()
    ids = {L: tok.encode(L, add_special_tokens=False)[0] for L in "ABCD"}

    def solve(prompt: str):
        chat = tok.apply_chat_template([{"role": "user", "content": prompt}],
                                       add_generation_prompt=True, tokenize=False)
        inputs = tok(chat, return_tensors="pt").to(llm.device)
        with torch.no_grad():
            logits = llm(**inputs).logits[0, -1]
        return max(ids, key=lambda L: float(logits[ids[L]]))
    return solve

solver = make_solver()
report = leakfilter.run_filter(list(pack), GOLD, ASR, solvers=[solver])
print(json.dumps(report.table(list(pack)), indent=2))

kept = leakfilter.apply_filter(list(pack), report)
ItemPack(kept, meta={**pack.meta, "leak_filtered": True}).save(
    f"{PACK_DIR}/item_pack.jsonl")
print(f"\n{len(pack)} -> {len(kept)} items survived the filter")

In [ ]:
# The floor check, and it is NOT "accuracy should be ~25%".
#
# The option set includes "not mentioned in the recording", so a model with no
# audio *should* choose it: abstaining is the correct response to being asked
# about a recording you cannot hear. The first real run scored 0.112 overall -
# 16 of 143, against exactly 14 null items - which is that behaviour, not a
# broken pack. Reading it as a failed 25% floor would have condemned a pack that
# was working.
#
# What actually indicates leakage is the model picking the CORRECT option on
# NON-NULL items above chance without hearing anything. That is the number to
# watch.
from collections import Counter

from undertone.protocol import question_only

roles = Counter()
non_null_correct = non_null_total = null_correct = null_total = 0
for item in pack:
    rendered = question_only(item, seed=SEED)
    letter = solver(rendered.prompt)
    role = rendered.letter_to_role.get(letter)
    roles[role] += 1
    if item.is_null:
        null_total += 1
        null_correct += role == "absent"
    else:
        non_null_total += 1
        non_null_correct += role == "correct"

leak = non_null_correct / max(1, non_null_total)
print(f"question-only role distribution: {dict(roles)}")
print(f"  non-null 'correct' rate: {non_null_correct}/{non_null_total} = {leak:.3f}"
      f"   (chance 0.25; ABOVE chance means the options leak the answer)")
print(f"  null-item abstention:    {null_correct}/{null_total}")
print(f"  abstention rate overall: {roles.get('absent', 0) / max(1, len(pack)):.3f}"
      " (high is expected and correct - it cannot hear the audio)")

if leak > 0.25:
    print("\nFAIL: the option text alone beats chance. Distractors are leaking; "
          "fix the item construction before running any model.")
else:
    print("\nOK: no audio, no signal - the options do not give the answer away.")

In [ ]:
# Verification: listen, confirm the key, mark it. Nothing is evidence until this
# has happened. Run it over the pack in batches; it is the slow step and there is
# no way around it.
import IPython.display as ipd, librosa

TO_REVIEW = list(pack)[:20]
for item in TO_REVIEW:
    path = os.path.join("/kaggle/working/item_pack", item.audio_path)
    audio, sr = librosa.load(path, sr=16000, mono=True,
                             offset=max(0, item.needle_start - 3),
                             duration=(item.needle_end - item.needle_start) + 6)
    print(f"\n{item.item_id}  [{item.category}]  {item.question}")
    for role in ("correct", "salience", "recency"):
        print(f"    {role:9s} {item.options[role]}")
    print(f"    provenance: {item.provenance['why']}")
    ipd.display(ipd.Audio(audio, rate=sr))

# After listening, record the verdicts and re-save:
#   VERDICTS = {"ES2002a_P1_000": True, "ES2002a_P3_004": False, ...}
VERDICTS = {}
if VERDICTS:
    kept = []
    for item in pack:
        if VERDICTS.get(item.item_id) is False:
            continue
        data = item.to_dict()
        data["provenance"] = {**item.provenance,
                              "verified": bool(VERDICTS.get(item.item_id, False))}
        kept.append(type(item).from_dict(data))
    ItemPack(kept, meta={**pack.meta, "verified": True}).save(
        "/kaggle/working/item_pack/item_pack.jsonl")
    print(f"{sum(1 for v in VERDICTS.values() if v)} verified, "
          f"{sum(1 for v in VERDICTS.values() if not v)} rejected")